In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings("ignore")
df = pd.read_csv('../data/raw/application_train.csv')
print(f"Data Loaded: {df.shape}")

Data Loaded: (307511, 122)


In [2]:
# Ratio Feature 
# Business rationale documented for each feature

# 1. Debt to Income ratio
# How much total debt is the borrower taking on relative to annual income ?
# Higher Ratio = Less repayment buffer = higher default risk

df['DEBT_TO_INCOME'] = df['AMT_CREDIT'] / df['AMT_INCOME_TOTAL']

# 2. Annuity to income
# What % of income goes towards loan repayment?
# Industry rule of thumb. Above 40% is high risk
df['ANNUITY_TO_INCOME'] = df['AMT_ANNUITY'] / df['AMT_INCOME_TOTAL'] 

# 3. Credit to Goods ratio
# How much of the purchase is being financed vs paid upfront?
# Ratio of 1.0 means 100% financed - no borrower equity
df['CREDIT_TO_GOODS'] = df['AMT_CREDIT'] / df['AMT_GOODS_PRICE']

# 4. Annuity to Credit Ration
# Implied loan tenure signal - lower ratio means longer repayment period
# Longer Tenure = more exposure to life event that cause defaulter 
df['ANNUITY_TO_CREDIT'] = df['AMT_ANNUITY'] / df['AMT_CREDIT']

# 5. Income per family member 
# Disposable income adjusted for dependents
df['INCOME_PER_PERSON'] = (df['AMT_INCOME_TOTAL'] / df['CNT_FAM_MEMBERS'].replace(0 , np.nan))

# 6. External Source Mean Score
# Average of three external credit score - reduce the noice of missingness
# More stable single signal than any individual score 
df['EXT_SOURCE_MEAN'] = df[['EXT_SOURCE_1' , 'EXT_SOURCE_2' , 'EXT_SOURCE_3']].mean(axis=1)

# 7. External Source Weighted Mean
# Ext - 3 showed highest correlation - give it more weight 
df['EXT_SOURCE_WEIGHTED'] = (
    0.5 * df['EXT_SOURCE_3'].fillna(df['EXT_SOURCE_3'].median()) + 
    0.3 * df['EXT_SOURCE_2'].fillna(df['EXT_SOURCE_2'].median()) +
    0.2 * df['EXT_SOURCE_1'].fillna(df['EXT_SOURCE_1'].median())
)

# Replace any infinity values with NaN across all new ratio features
ratio_cols = ['DEBT_TO_INCOME', 'ANNUITY_TO_INCOME', 'CREDIT_TO_GOODS',
              'ANNUITY_TO_CREDIT', 'INCOME_PER_PERSON',
              'EXT_SOURCE_MEAN', 'EXT_SOURCE_WEIGHTED']



print("Ratio features created successfully.")
print("\n New column added : 7")
print(f"\n Total column : {df.shape[1]}")

# Check for infinity first
for col in ratio_cols:
    inf_count = np.isinf(df[col]).sum()
    if inf_count > 0:
        print(f"{col}: {inf_count} infinity values found — replacing with NaN")
        df[col] = df[col].replace([np.inf, -np.inf], np.nan)

print("\nInfinity check complete")
df['DEBT_TO_INCOME'].head()

Ratio features created successfully.

 New column added : 7

 Total column : 129

Infinity check complete


0    2.007889
1    4.790750
2    2.000000
3    2.316167
4    4.222222
Name: DEBT_TO_INCOME, dtype: float64

In [3]:
# Validate ratio features — do they outperform raw columns?
raw_cols = ['AMT_CREDIT', 'AMT_INCOME_TOTAL', 
            'AMT_ANNUITY', 'AMT_GOODS_PRICE',
            'EXT_SOURCE_1', 'EXT_SOURCE_2', 'EXT_SOURCE_3']

new_cols = ['DEBT_TO_INCOME', 'ANNUITY_TO_INCOME',
            'CREDIT_TO_GOODS', 'ANNUITY_TO_CREDIT',
            'INCOME_PER_PERSON', 'EXT_SOURCE_MEAN',
            'EXT_SOURCE_WEIGHTED']

compare_cols = raw_cols + new_cols

correlations = df[compare_cols].corrwith(df['TARGET']).abs()
correlations = correlations.sort_values(ascending=False)

print("=== RAW vs ENGINEERED FEATURE CORRELATION ===\n")
for col in correlations.index:
    tag = " ← ENGINEERED" if col in new_cols else ""
    print(f"{col:<35} {correlations[col]:.4f}{tag}")

=== RAW vs ENGINEERED FEATURE CORRELATION ===

EXT_SOURCE_MEAN                     0.2221 ← ENGINEERED
EXT_SOURCE_WEIGHTED                 0.2192 ← ENGINEERED
EXT_SOURCE_3                        0.1789
EXT_SOURCE_2                        0.1605
EXT_SOURCE_1                        0.1553
CREDIT_TO_GOODS                     0.0694 ← ENGINEERED
AMT_GOODS_PRICE                     0.0396
AMT_CREDIT                          0.0304
ANNUITY_TO_INCOME                   0.0143 ← ENGINEERED
AMT_ANNUITY                         0.0128
ANNUITY_TO_CREDIT                   0.0127 ← ENGINEERED
DEBT_TO_INCOME                      0.0077 ← ENGINEERED
INCOME_PER_PERSON                   0.0066 ← ENGINEERED
AMT_INCOME_TOTAL                    0.0040


In [4]:
# Check default rate by DTI bands — reveals non-linear relationship
df['DTI_BAND'] = pd.cut(df['DEBT_TO_INCOME'],
                         bins=[0, 1, 2, 3, 4, 5, 100],
                         labels=['0-1x','1-2x','2-3x',
                                 '3-4x','4-5x','5x+'])

print("=== DEFAULT RATE BY DEBT-TO-INCOME BAND ===")
dti_default = df.groupby('DTI_BAND')['TARGET'].agg(['mean','count'])
dti_default.columns = ['Default Rate %', 'Count']
dti_default['Default Rate %'] = (dti_default['Default Rate %'] * 100).round(2)
print(dti_default.to_string())

=== DEFAULT RATE BY DEBT-TO-INCOME BAND ===
          Default Rate %  Count
DTI_BAND                       
0-1x                6.44  16174
1-2x                7.70  60164
2-3x                8.69  64858
3-4x                9.11  48703
4-5x                8.47  36805
5x+                 7.38  80807


In [5]:
# Confirm current state before Step 3.2
print(f"Total columns: {df.shape[1]}")
print(f"Expected: 129 (122 original + 7 ratio features)")
print(f"\nRatio features present:")

ratio_cols = ['DEBT_TO_INCOME', 'ANNUITY_TO_INCOME', 'CREDIT_TO_GOODS',
              'ANNUITY_TO_CREDIT', 'INCOME_PER_PERSON',
              'EXT_SOURCE_MEAN', 'EXT_SOURCE_WEIGHTED']

for col in ratio_cols:
    present = col in df.columns
    print(f"  {col:<30} {'✓' if present else '✗ MISSING'}")

Total columns: 130
Expected: 129 (122 original + 7 ratio features)

Ratio features present:
  DEBT_TO_INCOME                 ✓
  ANNUITY_TO_INCOME              ✓
  CREDIT_TO_GOODS                ✓
  ANNUITY_TO_CREDIT              ✓
  INCOME_PER_PERSON              ✓
  EXT_SOURCE_MEAN                ✓
  EXT_SOURCE_WEIGHTED            ✓


In [6]:
# ── IS_MISSING FLAG FEATURES ─────────────────────────────────────────────────
# For every column with medium or high missingness, create a companion binary
# flag that captures the signal contained in the absence of data.
# 0 = value was present in original data
# 1 = value was missing in original data

# Reload the missingness strategy we saved in Step 2.3

import pandas as pd

missing_strategy = pd.read_csv('../docs/missingness_strategy.csv', index_col=0)

# Get medium and high missingness columns

medium_cols = missing_strategy[missing_strategy["Severity"] == 'Medium(10-50%)'].index.tolist()

high_cols = missing_strategy[missing_strategy["Severity"] == 'High(>50%)'].index.tolist()

flag_target_cols = medium_cols + high_cols

print(f"Medium missingness columns: {len(medium_cols)}")
print(f"High missingness columns:   {len(high_cols)}")
print(f"Total IS_MISSING flags to create: {len(flag_target_cols)}")




Medium missingness columns: 16
High missingness columns:   41
Total IS_MISSING flags to create: 57


In [7]:
# Create IS_MISSING flag for every medium and high missingness column
flags_created = []

for col in flag_target_cols:
    flag_name = f"{col}_IS_MISSING"
    df[flag_name] = df[col].isnull().astype(int)
    flags_created.append(flag_name)

print(f"IS_MISSING flags created: {len(flags_created)}")
print(f"\nTotal columns now: {df.shape[1]}")
print(f"Expected: {129 + len(flags_created)}")

# Sanity check — verify a high missingness flag
print(f"\nSanity check — EXT_SOURCE_1_IS_MISSING:")
print(df['EXT_SOURCE_1_IS_MISSING'].value_counts())
print(f"Should show ~173,378 ones (56.38% of 307,511)")


IS_MISSING flags created: 57

Total columns now: 187
Expected: 186

Sanity check — EXT_SOURCE_1_IS_MISSING:
EXT_SOURCE_1_IS_MISSING
1    173378
0    134133
Name: count, dtype: int64
Should show ~173,378 ones (56.38% of 307,511)


In [8]:
# Identify the extra column
print(f"122 original + 7 ratio + 57 flags = {122 + 7 + 57}")
print(f"Actual columns: {df.shape[1]}")
print(f"\nExtra column is likely TOTAL_MISSING_COUNT")
print(f"TOTAL_MISSING_COUNT present: {'TOTAL_MISSING_COUNT' in df.columns}")


122 original + 7 ratio + 57 flags = 186
Actual columns: 187

Extra column is likely TOTAL_MISSING_COUNT
TOTAL_MISSING_COUNT present: False


In [9]:
# ── DAYS_EMPLOYED ANOMALY HANDLING ───────────────────────────────────────────
# 365,243 is a placeholder encoding for pensioners and unemployed applicants
# It is not a real employment tenure value
# Strategy: create a binary flag THEN replace with NaN for proper imputation

# Step A — Create the anomaly flag BEFORE replacing the value
df['DAYS_EMPLOYED_ANOMALY'] = (df['DAYS_EMPLOYED'] == 365243).astype(int)

# Step B — Replace 365,243 with NaN so it gets imputed with median
#           of the remaining legitimate employment values
df['DAYS_EMPLOYED'] = df['DAYS_EMPLOYED'].replace(365243, np.nan)

# Verify
print("=== DAYS_EMPLOYED ANOMALY HANDLING ===")
print(f"\nANOMALY flag — value counts:")
print(df['DAYS_EMPLOYED_ANOMALY'].value_counts())
print(f"\nDAYS_EMPLOYED after replacement:")
print(f"  Max value: {df['DAYS_EMPLOYED'].max()}")
print(f"  Missing values: {df['DAYS_EMPLOYED'].isnull().sum()}")
print(f"  Missing %: {df['DAYS_EMPLOYED'].isnull().mean()*100:.2f}%")
print(f"\nExpected: 55,374 ones in flag, 55,374 NaN in DAYS_EMPLOYED")

=== DAYS_EMPLOYED ANOMALY HANDLING ===

ANOMALY flag — value counts:
DAYS_EMPLOYED_ANOMALY
0    252137
1     55374
Name: count, dtype: int64

DAYS_EMPLOYED after replacement:
  Max value: 0.0
  Missing values: 55374
  Missing %: 18.01%

Expected: 55,374 ones in flag, 55,374 NaN in DAYS_EMPLOYED


In [10]:
# ── VALIDATE IS_MISSING FLAGS ─────────────────────────────────────────────────
# Check which flags carry the strongest signal by correlating with TARGET
# This confirms which absence patterns are most predictive of default

flag_correlations = df[flags_created].corrwith(df['TARGET']).abs()
flag_correlations = flag_correlations.sort_values(ascending=False)

print("=== TOP 15 IS_MISSING FLAGS BY CORRELATION WITH TARGET ===\n")
print(flag_correlations.head(15).round(4).to_string())

print(f"\n=== FLAGS WITH MEANINGFUL SIGNAL (correlation > 0.02) ===")
meaningful = flag_correlations[flag_correlations > 0.02]
print(f"Count: {len(meaningful)}")
print(meaningful.round(4).to_string())

print(f"\n=== FLAGS WITH WEAK SIGNAL (correlation <= 0.02) ===")
weak = flag_correlations[flag_correlations <= 0.02]
print(f"Count: {len(weak)}")

=== TOP 15 IS_MISSING FLAGS BY CORRELATION WITH TARGET ===

EMERGENCYSTATE_MODE_IS_MISSING             0.0414
TOTALAREA_MODE_IS_MISSING                  0.0412
ENTRANCES_MEDI_IS_MISSING                  0.0409
ENTRANCES_MODE_IS_MISSING                  0.0409
ENTRANCES_AVG_IS_MISSING                   0.0409
FLOORSMAX_AVG_IS_MISSING                   0.0408
FLOORSMAX_MODE_IS_MISSING                  0.0408
FLOORSMAX_MEDI_IS_MISSING                  0.0408
YEARS_BEGINEXPLUATATION_MEDI_IS_MISSING    0.0406
YEARS_BEGINEXPLUATATION_AVG_IS_MISSING     0.0406
YEARS_BEGINEXPLUATATION_MODE_IS_MISSING    0.0406
ELEVATORS_MEDI_IS_MISSING                  0.0403
ELEVATORS_AVG_IS_MISSING                   0.0403
ELEVATORS_MODE_IS_MISSING                  0.0403
APARTMENTS_AVG_IS_MISSING                  0.0403

=== FLAGS WITH MEANINGFUL SIGNAL (correlation > 0.02) ===
Count: 56
EMERGENCYSTATE_MODE_IS_MISSING             0.0414
TOTALAREA_MODE_IS_MISSING                  0.0412
ENTRANCES_MEDI_IS_MIS

In [11]:
# Identify the one weak flag
weak_flag = flag_correlations[flag_correlations <= 0.02]
print("Weak flag:")
print(weak_flag.round(4).to_string())
print(f"\nMissing % for this column:")
col_name = weak_flag.index[0].replace('_IS_MISSING', '')
print(f"{col_name}: {df[col_name].isnull().mean()*100:.2f}%")

Weak flag:
EXT_SOURCE_1_IS_MISSING    0.0186

Missing % for this column:
EXT_SOURCE_1: 56.38%


In [12]:
# ── FINAL CONFIRMATION ────────────────────────────────────────────────────────

# Confirm TOTAL_MISSING_COUNT exists and recalculate cleanly
# The original was calculated on raw data — recalculate on current dataframe
# using only the original 122 columns to avoid counting our own flags

original_cols = df.columns[:122].tolist()
df['TOTAL_MISSING_COUNT'] = df[original_cols].isnull().sum(axis=1)

print("=== TOTAL_MISSING_COUNT VALIDATION ===")
print(f"Correlation with TARGET: {df['TOTAL_MISSING_COUNT'].corr(df['TARGET']):.4f}")
print(f"\nAverage missing fields:")
comparison = df.groupby('TARGET')['TOTAL_MISSING_COUNT'].mean()
comparison.index = ['Repaid (0)', 'Defaulted (1)']
print(comparison.round(2))

print(f"\n=== STEP 3.2 FINAL COLUMN COUNT ===")
print(f"Original columns:          122")
print(f"Ratio features (Step 3.1):   7")
print(f"IS_MISSING flags:           57")
print(f"DAYS_EMPLOYED_ANOMALY:       1")
print(f"TOTAL_MISSING_COUNT:         1")
print(f"Expected total:            188")
print(f"Actual total:              {df.shape[1]}")

=== TOTAL_MISSING_COUNT VALIDATION ===
Correlation with TARGET: 0.0421

Average missing fields:
Repaid (0)       29.68
Defaulted (1)    32.92
Name: TOTAL_MISSING_COUNT, dtype: float64

=== STEP 3.2 FINAL COLUMN COUNT ===
Original columns:          122
Ratio features (Step 3.1):   7
IS_MISSING flags:           57
DAYS_EMPLOYED_ANOMALY:       1
TOTAL_MISSING_COUNT:         1
Expected total:            188
Actual total:              189


In [13]:
# Find the extra column
print(f"Expected 188, got {df.shape[1]} — finding the extra column")
print(f"\nLast 10 columns in dataframe:")
print(df.columns[-10:].tolist())

# Check if any column appears twice
duplicates = df.columns[df.columns.duplicated()].tolist()
print(f"\nDuplicate column names: {duplicates}")

# Check for any unexpected columns beyond our engineered set
expected_engineered = ratio_cols + flags_created + ['DAYS_EMPLOYED_ANOMALY', 'TOTAL_MISSING_COUNT']
all_engineered = [c for c in df.columns if c not in df.columns[:122]]
unexpected = [c for c in all_engineered if c not in expected_engineered]
print(f"\nUnexpected engineered columns: {unexpected}")
df.drop(columns=['DTI_BAND'], inplace=True)


Expected 188, got 189 — finding the extra column

Last 10 columns in dataframe:
['APARTMENTS_MODE_IS_MISSING', 'ENTRANCES_MODE_IS_MISSING', 'ENTRANCES_MEDI_IS_MISSING', 'ENTRANCES_AVG_IS_MISSING', 'LIVINGAREA_AVG_IS_MISSING', 'LIVINGAREA_MEDI_IS_MISSING', 'LIVINGAREA_MODE_IS_MISSING', 'HOUSETYPE_MODE_IS_MISSING', 'DAYS_EMPLOYED_ANOMALY', 'TOTAL_MISSING_COUNT']

Duplicate column names: []

Unexpected engineered columns: ['DTI_BAND']


In [14]:
# ── DAY COLUMN CONVERSIONS ────────────────────────────────────────────────────
# DAYS_BIRTH and DAYS_EMPLOYED are stored as negative integers
# Convert to positive years for interpretability and SHAP readability

# Age in years — negate and divide by 365
df['AGE_YEARS'] = (-df['DAYS_BIRTH'] / 365).round(1)

# Employment years — negate and divide by 365
# Note: DAYS_EMPLOYED already has 365,243 replaced with NaN from Step 3.2
# Those NaN values will remain NaN here — correct behaviour
df['EMPLOYMENT_YEARS'] = (-df['DAYS_EMPLOYED'] / 365).round(1)

# Days since ID was published — convert to years
# Older ID = more established identity = lower risk signal
df['ID_PUBLISH_YEARS'] = (-df['DAYS_ID_PUBLISH'] / 365).round(1)

# Days since last phone change — convert to years
# Longer time since change = more stable contact = lower risk signal
df['PHONE_CHANGE_YEARS'] = (-df['DAYS_LAST_PHONE_CHANGE'] / 365).round(1)

# Days since registration — convert to years
df['REGISTRATION_YEARS'] = (-df['DAYS_REGISTRATION'] / 365).round(1)

print("=== DAY COLUMN CONVERSIONS ===")
new_year_cols = ['AGE_YEARS', 'EMPLOYMENT_YEARS', 'ID_PUBLISH_YEARS',
                 'PHONE_CHANGE_YEARS', 'REGISTRATION_YEARS']

for col in new_year_cols:
    print(f"\n{col}:")
    print(f"  Min:  {df[col].min():.1f} years")
    print(f"  Max:  {df[col].max():.1f} years")
    print(f"  Mean: {df[col].mean():.1f} years")
    print(f"  NaN:  {df[col].isnull().sum()}")

=== DAY COLUMN CONVERSIONS ===

AGE_YEARS:
  Min:  20.5 years
  Max:  69.1 years
  Mean: 43.9 years
  NaN:  0

EMPLOYMENT_YEARS:
  Min:  0.0 years
  Max:  49.1 years
  Mean: 6.5 years
  NaN:  55374

ID_PUBLISH_YEARS:
  Min:  0.0 years
  Max:  19.7 years
  Mean: 8.2 years
  NaN:  0

PHONE_CHANGE_YEARS:
  Min:  -0.0 years
  Max:  11.8 years
  Mean: 2.6 years
  NaN:  1

REGISTRATION_YEARS:
  Min:  0.0 years
  Max:  67.6 years
  Mean: 13.7 years
  NaN:  0


In [15]:
# Fix floating point negative zero in PHONE_CHANGE_YEARS
df['PHONE_CHANGE_YEARS'] = df['PHONE_CHANGE_YEARS'].abs()

print(f"PHONE_CHANGE_YEARS min after fix: {df['PHONE_CHANGE_YEARS'].min()}")
print(f"Total columns: {df.shape[1]}")
print(f"Expected: 193 (188 + 5 new year columns)")

PHONE_CHANGE_YEARS min after fix: 0.0
Total columns: 193
Expected: 193 (188 + 5 new year columns)


In [16]:
# ── DOCUMENT COMPLIANCE SCORE ─────────────────────────────────────────────────
# Aggregate all 28 FLAG_DOCUMENT columns into a single compliance score
# Each flag = 1 if the applicant submitted that document, 0 if not
# Sum = total documents voluntarily submitted
# Higher score = more cooperative, organised, transparent borrower

# Identify all FLAG_DOCUMENT columns
doc_flag_cols = [c for c in df.columns if c.startswith('FLAG_DOCUMENT')]

print(f"FLAG_DOCUMENT columns found: {len(doc_flag_cols)}")
print(f"Columns: {doc_flag_cols}")

# Create the compliance score
df['DOCS_SUBMITTED_COUNT'] = df[doc_flag_cols].sum(axis=1)

print(f"\n=== DOCS_SUBMITTED_COUNT ===")
print(f"Min:  {df['DOCS_SUBMITTED_COUNT'].min()}")
print(f"Max:  {df['DOCS_SUBMITTED_COUNT'].max()}")
print(f"Mean: {df['DOCS_SUBMITTED_COUNT'].mean():.2f}")
print(f"NaN:  {df['DOCS_SUBMITTED_COUNT'].isnull().sum()}")

print(f"\nCorrelation with TARGET: "
      f"{df['DOCS_SUBMITTED_COUNT'].corr(df['TARGET']):.4f}")

print(f"\nDefault rate by document submission count:")
doc_default = df.groupby('DOCS_SUBMITTED_COUNT')['TARGET'].agg(['mean','count'])
doc_default.columns = ['Default Rate %', 'Count']
doc_default['Default Rate %'] = (doc_default['Default Rate %'] * 100).round(2)
print(doc_default.to_string())

FLAG_DOCUMENT columns found: 20
Columns: ['FLAG_DOCUMENT_2', 'FLAG_DOCUMENT_3', 'FLAG_DOCUMENT_4', 'FLAG_DOCUMENT_5', 'FLAG_DOCUMENT_6', 'FLAG_DOCUMENT_7', 'FLAG_DOCUMENT_8', 'FLAG_DOCUMENT_9', 'FLAG_DOCUMENT_10', 'FLAG_DOCUMENT_11', 'FLAG_DOCUMENT_12', 'FLAG_DOCUMENT_13', 'FLAG_DOCUMENT_14', 'FLAG_DOCUMENT_15', 'FLAG_DOCUMENT_16', 'FLAG_DOCUMENT_17', 'FLAG_DOCUMENT_18', 'FLAG_DOCUMENT_19', 'FLAG_DOCUMENT_20', 'FLAG_DOCUMENT_21']

=== DOCS_SUBMITTED_COUNT ===
Min:  0
Max:  4
Mean: 0.93
NaN:  0

Correlation with TARGET: 0.0172

Default rate by document submission count:
                      Default Rate %   Count
DOCS_SUBMITTED_COUNT                        
0                               5.52   29549
1                               8.45  270056
2                               4.64    7742
3                               9.82     163
4                             100.00       1


In [17]:
# Check if FLAG_DOCUMENT_1 exists
print(f"FLAG_DOCUMENT_1 exists: {'FLAG_DOCUMENT_1' in df.columns}")

# Check all FLAG columns to understand the full picture
all_flag_cols = [c for c in df.columns if c.startswith('FLAG_')]
print(f"\nAll FLAG columns: {len(all_flag_cols)}")
print(all_flag_cols)

FLAG_DOCUMENT_1 exists: False

All FLAG columns: 28
['FLAG_OWN_CAR', 'FLAG_OWN_REALTY', 'FLAG_MOBIL', 'FLAG_EMP_PHONE', 'FLAG_WORK_PHONE', 'FLAG_CONT_MOBILE', 'FLAG_PHONE', 'FLAG_EMAIL', 'FLAG_DOCUMENT_2', 'FLAG_DOCUMENT_3', 'FLAG_DOCUMENT_4', 'FLAG_DOCUMENT_5', 'FLAG_DOCUMENT_6', 'FLAG_DOCUMENT_7', 'FLAG_DOCUMENT_8', 'FLAG_DOCUMENT_9', 'FLAG_DOCUMENT_10', 'FLAG_DOCUMENT_11', 'FLAG_DOCUMENT_12', 'FLAG_DOCUMENT_13', 'FLAG_DOCUMENT_14', 'FLAG_DOCUMENT_15', 'FLAG_DOCUMENT_16', 'FLAG_DOCUMENT_17', 'FLAG_DOCUMENT_18', 'FLAG_DOCUMENT_19', 'FLAG_DOCUMENT_20', 'FLAG_DOCUMENT_21']


In [19]:
# ── CONTACT & ASSET FLAGS SCORE ───────────────────────────────────────────────
# Separate from document flags — captures reachability and asset ownership
# Higher score = more contactable, more assets = lower risk

contact_flag_cols = ['FLAG_MOBIL', 'FLAG_EMP_PHONE', 'FLAG_WORK_PHONE',
                     'FLAG_CONT_MOBILE', 'FLAG_PHONE', 'FLAG_EMAIL']

# Asset flags handled separately as they are strong individual predictors
# FLAG_OWN_CAR and FLAG_OWN_REALTY stay as individual features

df['CONTACT_FLAGS_COUNT'] = df[contact_flag_cols].sum(axis=1)

print("=== CONTACT_FLAGS_COUNT ===")
print(f"Min:  {df['CONTACT_FLAGS_COUNT'].min()}")
print(f"Max:  {df['CONTACT_FLAGS_COUNT'].max()}")
print(f"Mean: {df['CONTACT_FLAGS_COUNT'].mean():.2f}")
print(f"NaN:  {df['CONTACT_FLAGS_COUNT'].isnull().sum()}")
print(f"\nCorrelation with TARGET: "
      f"{df['CONTACT_FLAGS_COUNT'].corr(df['TARGET']):.4f}")

print(f"\nDefault rate by contact flag count:")
contact_default = df.groupby('CONTACT_FLAGS_COUNT')['TARGET'].agg(['mean','count'])
contact_default.columns = ['Default Rate %', 'Count']
contact_default['Default Rate %'] = (contact_default['Default Rate %']*100).round(2)
print(contact_default.to_string())

=== CONTACT_FLAGS_COUNT ===
Min:  1
Max:  6
Mean: 3.36
NaN:  0

Correlation with TARGET: 0.0208

Default rate by contact flag count:
                     Default Rate %   Count
CONTACT_FLAGS_COUNT                        
1                              3.70      27
2                              5.67   38357
3                              8.38  160972
4                              8.50   70671
5                              8.39   35492
6                              8.99    1992


In [20]:
# ── IS_YOUNG_BORROWER FLAG ────────────────────────────────────────────────────
# Applicants aged under 30 default at 11-11.74% vs portfolio average of 8.07%
# That is a 40% elevation above average — a distinct, targetable risk segment
# Binary flag isolates this segment explicitly for the model

df['IS_YOUNG_BORROWER'] = (df['AGE_YEARS'] < 30).astype(int)

print("=== IS_YOUNG_BORROWER FLAG ===")
print(f"\nValue counts:")
print(df['IS_YOUNG_BORROWER'].value_counts())
print(f"\nAs percentage:")
print(df['IS_YOUNG_BORROWER'].value_counts(normalize=True).round(4) * 100)

print(f"\nCorrelation with TARGET: "
      f"{df['IS_YOUNG_BORROWER'].corr(df['TARGET']):.4f}")

print(f"\nDefault rate by young borrower flag:")
young_default = df.groupby('IS_YOUNG_BORROWER')['TARGET'].agg(['mean','count'])
young_default.columns = ['Default Rate %', 'Count']
young_default['Default Rate %'] = (young_default['Default Rate %']*100).round(2)
young_default.index = ['30 and above (0)', 'Under 30 (1)']
print(young_default.to_string())

print(f"\nTotal columns now: {df.shape[1]}")
print(f"Expected: 195 (193 + CONTACT_FLAGS_COUNT + IS_YOUNG_BORROWER)")

=== IS_YOUNG_BORROWER FLAG ===

Value counts:
IS_YOUNG_BORROWER
0    262958
1     44553
Name: count, dtype: int64

As percentage:
IS_YOUNG_BORROWER
0    85.51
1    14.49
Name: proportion, dtype: float64

Correlation with TARGET: 0.0514

Default rate by young borrower flag:
                  Default Rate %   Count
30 and above (0)            7.50  262958
Under 30 (1)               11.47   44553

Total columns now: 196
Expected: 195 (193 + CONTACT_FLAGS_COUNT + IS_YOUNG_BORROWER)


In [21]:
# Find the extra column
print(f"Expected 195, got {df.shape[1]}")
print(f"\nAll engineered columns beyond original 122:")
all_engineered = df.columns[122:].tolist()
print(f"Count: {len(all_engineered)}")
print(all_engineered)

Expected 195, got 196

All engineered columns beyond original 122:
Count: 74
['DEBT_TO_INCOME', 'ANNUITY_TO_INCOME', 'CREDIT_TO_GOODS', 'ANNUITY_TO_CREDIT', 'INCOME_PER_PERSON', 'EXT_SOURCE_MEAN', 'EXT_SOURCE_WEIGHTED', 'FLOORSMAX_MODE_IS_MISSING', 'FLOORSMAX_AVG_IS_MISSING', 'FLOORSMAX_MEDI_IS_MISSING', 'YEARS_BEGINEXPLUATATION_MODE_IS_MISSING', 'YEARS_BEGINEXPLUATATION_MEDI_IS_MISSING', 'YEARS_BEGINEXPLUATATION_AVG_IS_MISSING', 'TOTALAREA_MODE_IS_MISSING', 'EMERGENCYSTATE_MODE_IS_MISSING', 'OCCUPATION_TYPE_IS_MISSING', 'EXT_SOURCE_3_IS_MISSING', 'AMT_REQ_CREDIT_BUREAU_HOUR_IS_MISSING', 'AMT_REQ_CREDIT_BUREAU_WEEK_IS_MISSING', 'AMT_REQ_CREDIT_BUREAU_MON_IS_MISSING', 'AMT_REQ_CREDIT_BUREAU_YEAR_IS_MISSING', 'AMT_REQ_CREDIT_BUREAU_DAY_IS_MISSING', 'AMT_REQ_CREDIT_BUREAU_QRT_IS_MISSING', 'COMMONAREA_AVG_IS_MISSING', 'COMMONAREA_MODE_IS_MISSING', 'COMMONAREA_MEDI_IS_MISSING', 'NONLIVINGAPARTMENTS_MEDI_IS_MISSING', 'NONLIVINGAPARTMENTS_MODE_IS_MISSING', 'NONLIVINGAPARTMENTS_AVG_IS_MISSING'

In [22]:
# ── STEP 3.3 CORRELATION SUMMARY ─────────────────────────────────────────────
step_33_features = [
    'AGE_YEARS', 'EMPLOYMENT_YEARS', 'ID_PUBLISH_YEARS',
    'PHONE_CHANGE_YEARS', 'REGISTRATION_YEARS',
    'DOCS_SUBMITTED_COUNT', 'CONTACT_FLAGS_COUNT',
    'IS_YOUNG_BORROWER'
]

print("=== STEP 3.3 FEATURES — CORRELATION WITH TARGET ===\n")
corr = df[step_33_features].corrwith(df['TARGET']).abs()
corr = corr.sort_values(ascending=False)

for col in corr.index:
    bar = '█' * int(corr[col] * 500)
    print(f"{col:<30} {corr[col]:.4f}  {bar}")

print(f"\nTotal columns: {df.shape[1]}")
print(f"Confirmed: 196")

=== STEP 3.3 FEATURES — CORRELATION WITH TARGET ===

AGE_YEARS                      0.0782  ███████████████████████████████████████
EMPLOYMENT_YEARS               0.0750  █████████████████████████████████████
PHONE_CHANGE_YEARS             0.0552  ███████████████████████████
ID_PUBLISH_YEARS               0.0515  █████████████████████████
IS_YOUNG_BORROWER              0.0514  █████████████████████████
REGISTRATION_YEARS             0.0420  ████████████████████
CONTACT_FLAGS_COUNT            0.0208  ██████████
DOCS_SUBMITTED_COUNT           0.0172  ████████

Total columns: 196
Confirmed: 196
